# 05 - Mapping And SLAM Intro

## What / Why / How

**What we are trying to do:** Build intuition for mapping and localization with occupancy grids, log odds, and a particle filter.

**Why this matters:** Navigation requires knowing both where obstacles are and where the robot is. SLAM grows from these ideas.

**How we will do it:** Update grid cells from simulated range hits, then use landmark distance measurements to localize a robot with particles.

## Goal

Mapping and SLAM connect robot motion with world structure.

This notebook introduces:

- Occupancy grids
- Log-odds updates
- Particle-filter localization

Full SLAM is more complex, but these pieces are the foundation.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## Occupancy Grid With Log Odds

In [ ]:
grid_size = 40
resolution = 0.1
log_odds = np.zeros((grid_size, grid_size))
robot_cell = np.array([20, 20])

# Simulated obstacle points in grid cells.
obstacles = np.array([
    [28, 24], [30, 25], [31, 26], [12, 28], [10, 29], [25, 10]
])

def update_ray(log_odds, robot_cell, hit_cell, free_delta=-0.4, occ_delta=0.9):
    steps = int(np.linalg.norm(hit_cell - robot_cell)) + 1
    for alpha in np.linspace(0.0, 1.0, steps):
        cell = np.round(robot_cell + alpha * (hit_cell - robot_cell)).astype(int)
        r, c = cell
        if 0 <= r < log_odds.shape[0] and 0 <= c < log_odds.shape[1]:
            log_odds[r, c] += free_delta
    r, c = hit_cell
    if 0 <= r < log_odds.shape[0] and 0 <= c < log_odds.shape[1]:
        log_odds[r, c] += occ_delta

for hit in obstacles:
    update_ray(log_odds, robot_cell, hit)

occupancy_prob = 1 - 1 / (1 + np.exp(log_odds))
print("occupied-ish cells:", np.argwhere(occupancy_prob > 0.6)[:10])

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    plt.imshow(occupancy_prob, origin="lower", cmap="gray_r", vmin=0, vmax=1)
    plt.scatter(robot_cell[1], robot_cell[0], c="tab:blue", label="robot")
    plt.colorbar(label="P(occupied)")
    plt.legend()
    plt.title("Occupancy grid")
    plt.show()
else:
    plot_unavailable()

## Particle Filter Localization In A 1D Hallway

In [ ]:
rng = np.random.default_rng(42)
landmarks = np.array([1.5, 5.5, 8.5])
true_x = 3.2
sigma = 0.18
measurements = np.abs(landmarks - true_x) + rng.normal(0, sigma, size=len(landmarks))

num_particles = 2000
particles = rng.uniform(0, 10, size=num_particles)
weights = np.ones(num_particles) / num_particles

for iteration in range(6):
    expected = np.abs(particles[:, None] - landmarks[None, :])
    residual = expected - measurements[None, :]
    weights = np.exp(-0.5 * np.sum((residual / sigma) ** 2, axis=1))
    weights += 1e-12
    weights /= weights.sum()
    particles = rng.choice(particles, size=num_particles, p=weights)
    particles += rng.normal(0, 0.04, size=num_particles)
    particles = np.clip(particles, 0, 10)

estimate = particles.mean()
print("true x:", true_x)
print("estimated x:", round(float(estimate), 3))
print("particle std:", round(float(particles.std()), 3))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(8, 2.5))
    plt.hist(particles, bins=50, density=True, alpha=0.7)
    for lm in landmarks:
        plt.axvline(lm, color="black", linestyle=":", alpha=0.4)
    plt.axvline(true_x, color="tab:green", label="true")
    plt.axvline(estimate, color="tab:red", label="estimate")
    plt.xlabel("x position")
    plt.yticks([])
    plt.legend()
    plt.title("Particle belief")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add more obstacle hits to the occupancy grid.
2. Increase sensor noise in the particle filter.
3. Move the true position close to a symmetric landmark layout and observe ambiguity.